# My personal break habits

This notebook explores **actual completed GPX recordings**. A row in the training table is one recording, not one GPS point. The moving-time predictor stays separate.

First run `python -m hiking_predictor analyze-breaks` from the project root. That command reuses the existing importer, writes auditable break labels, preserves corrections, and saves the independent break model. This notebook reads those exports and repeats the small whole-hike evaluation in memory.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
from hiking_time_predictor.break_analysis import break_figures
from hiking_time_predictor.break_model import (BASELINE, break_coefficients, load_break_model, train_break_predictor)
from hiking_time_predictor.web_breaks import load_break_history

PROJECT = Path.cwd()
if not (PROJECT / "src/hiking_time_predictor").exists():
    PROJECT = PROJECT.parent
if not (PROJECT / "data/processed/break_events.csv").exists():
    raise FileNotFoundError("Run python -m hiking_predictor analyze-breaks from the project root first.")
hikes, events, saved_predictions = load_break_history(PROJECT)
artifact = load_break_model(PROJECT / "models/break_predictor.joblib", project=PROJECT)
if artifact.minutes_per_mile is None:
    raise ValueError(artifact.reason)
print(artifact.reason)
display(pd.Series(json.loads((PROJECT / "data/processed/break_analysis/settings.json").read_text())["break_config"], name="Break thresholds"))

## What counts as a break?

The existing GPX code uses timestamps, distance, speed, and sustained geographic clusters to classify movement, stationary time, and unknown intervals. We combine contiguous intervals into events, require a meaningful duration, and reject drifting unconfirmed clusters. An unconfirmed GPS gap remains unknown even if its endpoints match. Explicit hiker confirmations retain their provenance. A confirmation for one interval does not confirm neighboring ambiguous intervals.

Short stops, unknown intervals, camping candidates, and reviewed non-hiking activity are excluded from ordinary rest labels. These are probable stationary rests, not proven intentional breaks. Check the quality flags and review labels.

In [ ]:
display(events.groupby("break_classification").duration_minutes.agg(["count", "sum"]))
ordinary = events.loc[events.break_classification.eq("stationary_break")]
miles = hikes.distance_miles.sum()
findings = pd.Series({
    "recordings": len(hikes), "measured_miles": miles,
    "ordinary_break_count": len(ordinary), "total_break_minutes": ordinary.duration_minutes.sum(),
    "mean_break_duration_min": ordinary.duration_minutes.mean(),
    "median_break_duration_min": ordinary.duration_minutes.median(),
    "pooled_break_minutes_per_mile": ordinary.duration_minutes.sum() / miles,
    "pooled_breaks_per_mile": len(ordinary) / miles,
    "mean_spacing_miles": ordinary.groupby("hike_id").distance_into_hike_miles.diff().mean(),
})
display(findings)
display(hikes)

## Explore the actual measurements

The six figures show distance, elevation, individual durations, break frequency, locations along each recording, and moving time versus rest time. Location rows use the same order as the hike summary above. Larger markers indicate longer rests. A correlation describes association in this small history; it does not show that climbing caused a stop. Blank hike types stay unknown.

In [ ]:
for name, figure in break_figures(hikes, events).items():
    print(name)
    display(figure)
display(hikes[["distance_miles", "elevation_gain_ft", "total_break_minutes"]].corr())
print("Known hike types:", int(hikes.hike_type.isin(["day_hike", "backpacking"]).sum()), "of", len(hikes))

## Start with a personal baseline

A **feature** is an input known before a hike. A **target** is the number we want to learn: total ordinary break minutes. The baseline estimates a personal rate:

$$r = \frac{\text{eligible historical break minutes}}{\text{eligible historical miles}}, \qquad \hat{b}=r\times\text{planned miles}.$$

This is a pooled rate, not the average of per-hike ratios. Eligible recordings exclude excessive unknown time, incomplete timestamps, camping and non-hiking activity. The eligible rate may be biased toward better-recorded trips. Hikes with zero observed breaks remain valid examples.

In [ ]:
eligible = hikes.loc[hikes.training_eligible]
personal_rate = eligible.total_break_minutes.sum() / eligible.distance_miles.sum()
print(f"Eligible independent recordings: {len(eligible)}")
print(f"Personal baseline: {personal_rate:.4f} ordinary break minutes per measured mile")
print(f"Pooled rate across all recordings: {findings['pooled_break_minutes_per_mile']:.4f} min/mi")

## Test regression without leaking the answer

Simple linear regression A finds an intercept and a distance slope: $\hat b = a + w_d d$. Multiple regression B adds gain and loss: $\hat b = a + w_d d + w_g g + w_l l$. The intercept is predicted minutes at zero inputs; a slope is the change in predicted minutes for one extra mile or foot, with other inputs held fixed. These fitted numbers are associations, not physiological laws. Negative predictions are clamped to zero.

Model C adds a categorical day-hike/backpacking input through one-hot encoding, only with enough known examples of both types. We never use actual moving or elapsed time, observed break count, or actual break duration as inputs.

**Leave-one-hike-out validation** withholds a whole recording, computes the baseline and fits regression on the other recordings, then predicts the withheld target. This repeats for every eligible hike. No part of a held-out hike enters its training fold. The final saved model is fitted on all eligible hikes after evaluation. MAE is the average absolute error in minutes. Too few hikes means no score; fewer than twelve eligible hikes keeps regression experimental even if it has a lower error.

In [ ]:
repeated_artifact, held_out = train_break_predictor(hikes, artifact.config)
for fold in repeated_artifact.evaluation["folds"]:
    assert set(fold["training_hike_ids"]).isdisjoint(fold["test_hike_ids"])
print(repeated_artifact.reason)
print(repeated_artifact.evaluation["model_c_status"])
display(pd.DataFrame(repeated_artifact.evaluation["metrics"]))
display(break_coefficients(repeated_artifact))
if held_out.empty:
    print("MAE unavailable: too few eligible independent hikes.")
else:
    display(held_out)
    display(break_figures(hikes, events, held_out)["actual_predicted"])
    display(pd.DataFrame(repeated_artifact.evaluation["folds"]))

## Your existing planned route

The same Python functions used by Streamlit calculate moving time, a separate personal break estimate, and their sum. Breaks are added once after moving-segment predictions. Out and back uses the full two-leg mileage. The current primary moving estimate may remain Naismith while a saved moving regression is experimental; both are labeled in the report. This exported example uses your existing untimed Hogpen GPX and does not add it to training.

In [ ]:
example_path = PROJECT / "data/processed/break_analysis/hogpen_example.csv"
if example_path.exists():
    example = pd.read_csv(example_path)
    display(example[["direction", "distance_miles", "moving_minutes", "break_minutes", "total_minutes", "experimental_total_minutes"]])
    assert ((example.moving_minutes + example.break_minutes - example.total_minutes).abs() < 1e-8).all()
else:
    print("No unique existing Hogpen plan was available when the analysis ran.")

## Review and limitations

Use **My Hiking Data → MY BREAKING HABITS → Review and correct detected breaks**, or edit `data/break_corrections.csv` using exact hike IDs and UTC event endpoints. Corrections live separately, survive reprocessing, and trigger an error if changed thresholds make their endpoints stale. Rerun `analyze-breaks` after editing. Measured moving intervals must first be confirmed as rests and reimported before being counted as breaks.

The small number of independent recordings, unknown hike types, uncertain GPS classification, and incomplete mileage limit the analysis. Confirmed long gaps depend on your review. Short pauses and unresolved time are omitted, so the target is ordinary meaningful rest time rather than every nonmoving minute. Stationary time at a trailhead may need manual exclusion. Consolidate recordings from the same outing before treating them as independent hikes. Geometry on untimed future plans may differ from historical moving mileage. Weather, fatigue, pack weight, and reasons for stopping are not learned here. MAE is not a future error guarantee.